# Data Splitting

## Imports

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from scipy import sparse

model_data_path = "../data/modeling_data/"
splits_path = "../data/splits/"

os.makedirs(splits_path + "7030", exist_ok=True)
os.makedirs(splits_path + "5050", exist_ok=True)
os.makedirs(splits_path + "3070", exist_ok=True)

def save_split(X_train, X_test, y_train, y_test, data, folder):
    out_path = splits_path + folder + "/"
    os.makedirs(out_path, exist_ok=True)

    # ---- Save X_train ----
    if sparse.issparse(X_train):  
        # Sparse matrix
        sparse.save_npz(out_path + f"X_train_{data}.npz", X_train)
    else:
        # DataFrame or ndarray
        pd.DataFrame(X_train).to_csv(out_path + f"X_train_{data}.csv", index=False)

    # ---- Save X_test ----
    if sparse.issparse(X_test):
        sparse.save_npz(out_path + f"X_test_{data}.npz", X_test)
    else:
        pd.DataFrame(X_test).to_csv(out_path + f"X_test_{data}.csv", index=False)

    # ---- Save y labels ----
    y_train.to_csv(out_path + f"y_train_{data}.csv", index=False)
    y_test.to_csv(out_path + f"y_test_{data}.csv", index=False)

    print(f"Saved split to {out_path}")

def manual_stratified_split(X, y, train_ratio, seed=42):
    np.random.seed(seed)
    y_arr = np.array(y)

    train_idx = []
    test_idx = []

    # Stratified split
    for cls in np.unique(y_arr):
        cls_idx = np.where(y_arr == cls)[0]
        np.random.shuffle(cls_idx)

        n_train = int(len(cls_idx) * train_ratio)
        train_idx.extend(cls_idx[:n_train])
        test_idx.extend(cls_idx[n_train:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)

    # Slice, but DO NOT reset index yet
    if isinstance(X, pd.DataFrame):
        X_train = X.iloc[train_idx]
        X_test  = X.iloc[test_idx]
    else:
        X_train = X[train_idx]
        X_test  = X[test_idx]

    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    return X_train, X_test, y_train, y_test, train_idx, test_idx





## Splitting the Wine Dataset:

In [2]:
wine = pd.read_csv(model_data_path + "wine_combined_data_model.csv")
y = wine["is_red"]
X = wine.drop(columns=["is_red"])
print("Class distribution (0 = white, 1 = red):")
print(y.value_counts())
print("\nPercentage:")
print(y.value_counts(normalize=True))



Class distribution (0 = white, 1 = red):
is_red
0    4898
1    1599
Name: count, dtype: int64

Percentage:
is_red
0    0.753886
1    0.246114
Name: proportion, dtype: float64


Not Balanced: Need to do Stratified Sampling

In [3]:
X_train_7030, X_test_7030, y_train_7030, y_test_7030, idx_train_7030, idx_test_7030 = \
    manual_stratified_split(X, y, 0.70)

save_split(X_train_7030, X_test_7030, y_train_7030, y_test_7030, "wine", "7030")

print("7030 leakage:", len(set(idx_train_7030) & set(idx_test_7030)))

X_train_3070, X_test_3070, y_train_3070, y_test_3070, idx_train_3070, idx_test_3070 = \
    manual_stratified_split(X, y, 0.30)

save_split(X_train_3070, X_test_3070, y_train_3070, y_test_3070, "wine", "3070")

print("3070 leakage:", len(set(idx_train_3070) & set(idx_test_3070)))


X_train_5050, X_test_5050, y_train_5050, y_test_5050, idx_train_5050, idx_test_5050 = \
    manual_stratified_split(X, y, 0.50)

save_split(X_train_5050, X_test_5050, y_train_5050, y_test_5050, "wine", "5050")

print("5050 leakage:", len(set(idx_train_5050) & set(idx_test_5050)))


Saved split to ../data/splits/7030/
7030 leakage: 0
Saved split to ../data/splits/3070/
3070 leakage: 0
Saved split to ../data/splits/5050/
5050 leakage: 0


## Spliting Customer

In [4]:
customer = pd.read_csv(model_data_path + "customer_features_model.csv")
target_col = "HighValueCustomer"

y = customer[target_col]
X = customer.drop(columns=[target_col])

print("Counts:")
print(y.value_counts())

print("\nPercentages:")
print(y.value_counts(normalize=True))



Counts:
HighValueCustomer
1    1099
0     913
Name: count, dtype: int64

Percentages:
HighValueCustomer
1    0.546223
0    0.453777
Name: proportion, dtype: float64


Still a bit inbalanced --> stratified

In [5]:
X_train_7030, X_test_7030, y_train_7030, y_test_7030, idx_train_7030, idx_test_7030 = \
    manual_stratified_split(X, y, 0.70)

save_split(X_train_7030, X_test_7030, y_train_7030, y_test_7030, "customer", "7030")

print("Customer 7030 leakage:", len(set(idx_train_7030) & set(idx_test_7030)))

X_train_3070, X_test_3070, y_train_3070, y_test_3070, idx_train_3070, idx_test_3070 = \
    manual_stratified_split(X, y, 0.30)

save_split(X_train_3070, X_test_3070, y_train_3070, y_test_3070, "customer", "3070")

print("Customer 3070 leakage:", len(set(idx_train_3070) & set(idx_test_3070)))

X_train_5050, X_test_5050, y_train_5050, y_test_5050, idx_train_5050, idx_test_5050 = \
    manual_stratified_split(X, y, 0.50)

save_split(X_train_5050, X_test_5050, y_train_5050, y_test_5050, "customer", "5050")

print("Customer 5050 leakage:", len(set(idx_train_5050) & set(idx_test_5050)))


Saved split to ../data/splits/7030/
Customer 7030 leakage: 0
Saved split to ../data/splits/3070/
Customer 3070 leakage: 0
Saved split to ../data/splits/5050/
Customer 5050 leakage: 0


## Splitting Cup98

In [6]:

X = sparse.load_npz(model_data_path + "cup98_features.npz")
# Load labels
y = pd.read_csv(model_data_path + "cup98_labels.csv")["TARGET_B"]
print("Counts:")
print(y.value_counts())

print("\nPercentages:")
print(y.value_counts(normalize=True))


Counts:
TARGET_B
0    90569
1     4843
Name: count, dtype: int64

Percentages:
TARGET_B
0    0.949241
1    0.050759
Name: proportion, dtype: float64


Worst distribution --> stratify

In [7]:
X_train_7030, X_test_7030, y_train_7030, y_test_7030, idx_train_7030, idx_test_7030 = \
    manual_stratified_split(X, y, 0.70)

save_split(X_train_7030, X_test_7030, y_train_7030, y_test_7030, "cup98", "7030")

print("CUP98 7030 leakage:", len(set(idx_train_7030) & set(idx_test_7030)))

X_train_3070, X_test_3070, y_train_3070, y_test_3070, idx_train_3070, idx_test_3070 = \
    manual_stratified_split(X, y, 0.30)

save_split(X_train_3070, X_test_3070, y_train_3070, y_test_3070, "cup98", "3070")

print("CUP98 3070 leakage:", len(set(idx_train_3070) & set(idx_test_3070)))

X_train_5050, X_test_5050, y_train_5050, y_test_5050, idx_train_5050, idx_test_5050 = \
    manual_stratified_split(X, y, 0.50)

save_split(X_train_5050, X_test_5050, y_train_5050, y_test_5050, "cup98", "5050")

print("CUP98 5050 leakage:", len(set(idx_train_5050) & set(idx_test_5050)))


Saved split to ../data/splits/7030/
CUP98 7030 leakage: 0
Saved split to ../data/splits/3070/
CUP98 3070 leakage: 0
Saved split to ../data/splits/5050/
CUP98 5050 leakage: 0


In [8]:
import shutil

shutil.rmtree(model_data_path)